In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings

# ==========================================================
# SETTINGS
# ==========================================================

# Main EPM folder (adjust to your project path)
main_dir = Path("/Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/EPM").resolve()
timepoints = ["Preinduction", "W1", "W2", "W3"]

# EthoVision zone labels
target_zone_labels = [
    "In zone 2(O1_prevC1 / Center-point)",
    "In zone 2(O2_prevC2 / Center-point)",
    "In zone 2(C1_prevO1 / Center-point)",
    "In zone 2(C2_prevO2 / Center-point)",
    "In zone 2(NeutralZone / Center-point)",
]

# Fixed colors
zone_colors = {
    "In zone 2(O1_prevC1 / Center-point)": "#E41A1C",  # red
    "In zone 2(O2_prevC2 / Center-point)": "#377EB8",  # blue
    "In zone 2(C1_prevO1 / Center-point)": "#4DAF4A",  # green
    "In zone 2(C2_prevO2 / Center-point)": "#984EA3",  # purple
    "In zone 2(NeutralZone / Center-point)": "#FF7F00",# orange
}

video_fps = 50.0
fp_fps = 50.0

# Figures directory
figures_main_dir = main_dir / "Figures"
figures_main_dir.mkdir(exist_ok=True, parents=True)

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

def _mouse_genotype_tag(mouse_id: str) -> str | None:
    id_clean = mouse_id.strip()
    if id_clean in {"372", "376", "423"}:
        return "NE"
    if id_clean in {"374", "429"}:
        return "WT"
    return None

def _read_export_and_get_fp_and_mouse_id(xl_path: Path):
    df = pd.read_excel(xl_path, header=None)
    key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()

    match_fp = df.loc[key_col == "fp file"]
    if match_fp.empty:
        raise ValueError(f"'FP file' row not found in {xl_path.name}")
    fp_id = str(match_fp.iloc[0, 1]).strip()

    match_id = df.loc[key_col == "id"]
    if match_id.empty:
        raise ValueError(f"'ID' row not found in {xl_path.name}")
    mouse_id = str(match_id.iloc[0, 1]).strip()

    return fp_id, mouse_id


# def _read_export_and_get_fp_and_mouse_id(xl_path: Path) -> tuple[str, str]:
#     df = pd.read_excel(xl_path, header=None)
#     key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()
#     match_fp = df[key_col == "fp file"]
#     if match_fp.empty:
#         raise ValueError(f"'FP file' row not found in {xl_path.name}")
#     fp_id = str(df.iloc[match_fp.index[0], 1]).strip()
#     if not fp_id or fp_id.lower() == "nan":
#         raise ValueError(f"Empty FP id in {xl_path.name}")
#     match_id = df[key_col == "id"]
#     if match_id.empty:
#         raise ValueError(f"'ID' row not found in {xl_path.name}")
#     mouse_id = str(df.iloc[match_id.index[0], 1]).strip()
#     if not mouse_id or mouse_id.lower() == "nan":
#         raise ValueError(f"Empty mouse ID in {xl_path.name}")
#     return fp_id, mouse_id

def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str) -> tuple[Path, Path]:
    fp_candidates = list(fp_folder.glob(f"{fp_id}.csv"))
    if not fp_candidates:
        fp_candidates = [p for p in fp_folder.glob("*.csv") if fp_id.lower() in p.name.lower()]
    if not fp_candidates:
        raise FileNotFoundError(f"No FP csv in {fp_folder} for id {fp_id}")
    fp_csv = sorted(fp_candidates)[0]
    ts_candidates = list(fp_folder.glob("timestamp*.csv"))
    if not ts_candidates:
        ts_candidates = [p for p in fp_folder.glob("*.csv") if "time" in p.name.lower()]
    if not ts_candidates:
        raise FileNotFoundError(f"No timestamp csv in {fp_folder}")
    ts_csv = sorted(ts_candidates)[0]
    return fp_csv, ts_csv

def _coerce_bool_col(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int).astype(bool)
    low = series.astype(str).str.strip().str.lower()
    return low.isin(["true", "1", "t", "yes", "y"])

def _get_video_window(ts: pd.DataFrame) -> tuple[float, float]:
    state_col = "DigitalIOState" if "DigitalIOState" in ts.columns else next(
        c for c in ts.columns if "digital" in c.lower() or "state" in c.lower()
    )
    time_col = "SystemTimestamp" if "SystemTimestamp" in ts.columns else next(
        c for c in ts.columns if "time" in c.lower()
    )
    state = _coerce_bool_col(ts[state_col])
    t = pd.to_numeric(ts[time_col], errors="coerce")
    if state.sum() == 0 or (~state).sum() == 0:
        raise ValueError("Timestamp must contain at least one True and one False in DigitalIOState")
    video_start = float(t[state].iloc[0])
    video_stop = float(t[~state].iloc[0])
    return video_start, video_stop

def _build_time_vector_from_fp(fp_df: pd.DataFrame) -> np.ndarray:
    if "SystemTimestamp" in fp_df.columns:
        tt = pd.to_numeric(fp_df["SystemTimestamp"], errors="coerce").to_numpy()
        if np.isfinite(tt).sum() >= len(tt) * 0.8:
            return tt
    return np.arange(len(fp_df), dtype=float) / fp_fps

def _snap_down_index(t: np.ndarray, target: float) -> int:
    return max(0, np.searchsorted(t, target, side="right") - 1)

def _trim_fp_to_window(fp_df: pd.DataFrame, start: float, stop: float) -> pd.DataFrame:
    tt = _build_time_vector_from_fp(fp_df)
    if len(tt) != len(fp_df):
        n = min(len(tt), len(fp_df))
        tt = tt[:n]
        fp_df = fp_df.iloc[:n, :].reset_index(drop=True)
    i0 = _snap_down_index(tt, start)
    i1 = _snap_down_index(tt, stop)
    i1 = max(i1, i0)
    out = fp_df.iloc[i0:i1+1, :].copy()
    out["Time_video"] = tt[i0:i1+1] - tt[i0]
    return out

_norm = re.compile(r"[^a-z0-9]+")
def _norm_key(s: str) -> str:
    return _norm.sub("", s.lower())

def _load_ethovision_sheet_with_targets(eth_path: Path, targets: list[str], video_fps: float):
    preview = pd.read_excel(eth_path, usecols=[0], nrows=100, header=None)
    header_row = preview[preview.iloc[:, 0].astype(str).str.contains("Trial time", case=False, na=False)].index
    if len(header_row) == 0:
        raise ValueError(f"Could not find 'Trial time' in {eth_path.name}")
    header_row = header_row[0]
    eth = pd.read_excel(eth_path, header=header_row).dropna(axis=1, how="all")
    norm_map = {_norm_key(c): c for c in eth.columns}
    matched_cols = [norm_map[_norm_key(t)] for t in targets if _norm_key(t) in norm_map]
    if not matched_cols:
        raise ValueError(f"No zone columns found in {eth_path.name}")
    time_col = next((c for c in eth.columns if "time" in str(c).lower()), None)
    if time_col is None:
        eth["Time_s"] = np.arange(len(eth)) / video_fps
    else:
        eth["Time_s"] = pd.to_numeric(eth[time_col], errors="coerce")
    return eth[["Time_s"] + matched_cols].copy(), matched_cols

def _ethogram_segments(eth: pd.DataFrame, beh_cols: list[str]) -> pd.DataFrame:
    segs = []
    for beh in beh_cols:
        mask = (eth[beh] == 1)
        if mask.sum() == 0:
            continue
        diff = mask.astype(int).diff().fillna(0)
        starts = eth.loc[diff == 1, "Time_s"].values
        ends = eth.loc[diff == -1, "Time_s"].values
        if mask.iloc[0]:
            starts = np.insert(starts, 0, eth["Time_s"].iloc[0])
        if mask.iloc[-1]:
            ends = np.append(ends, eth["Time_s"].iloc[-1])
        for s, e in zip(starts, ends):
            segs.append({"Behavior": beh, "Start_time_s": float(s), "End_time_s": float(e), "Duration_s": float(e - s)})
    if not segs:
        return pd.DataFrame(columns=["Behavior", "Start_time_s", "End_time_s", "Duration_s"])
    return pd.DataFrame(segs).sort_values(["Behavior", "Start_time_s"]).reset_index(drop=True)

def _plot_trial(fp_trim: pd.DataFrame, eth_timeline: pd.DataFrame, beh_cols: list[str], out_png: Path, title_prefix: str):
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios": [2,2,1]})
    if "LedState" not in fp_trim.columns:
        raise ValueError("FP csv must contain 'LedState' with values 1 and 2")
    chan = "G0" if "G0" in fp_trim.columns else next(
        c for c in fp_trim.columns if pd.api.types.is_numeric_dtype(fp_trim[c]) and c not in {"SystemTimestamp","LedState","Time_video","Time_s","Timestamp"}
    )
    fp1 = fp_trim[fp_trim["LedState"]==1]
    axes[0].plot(fp1["Time_video"], fp1[chan], linewidth=0.8, color="gray")
    axes[0].set_ylabel(f"{chan} (LED 1 Isosbestic)")
    axes[0].set_title(f"{title_prefix}: LED 1")
    axes[0].grid(True)
    fp2 = fp_trim[fp_trim["LedState"]==2]
    axes[1].plot(fp2["Time_video"], fp2[chan], linewidth=0.8, color="green")
    axes[1].set_ylabel(f"{chan} (LED 2 GCaMP)")
    axes[1].set_title(f"{title_prefix}: LED 2")
    axes[1].grid(True)
    if eth_timeline is not None and not eth_timeline.empty:
        for i, beh in enumerate(beh_cols):
            sub = eth_timeline[eth_timeline["Behavior"]==beh]
            color = zone_colors.get(beh, f"C{i}")
            for _, row in sub.iterrows():
                axes[2].barh(i, width=row["Duration_s"], left=row["Start_time_s"], height=0.6, color=color)
        axes[2].set_yticks(range(len(beh_cols)))
        axes[2].set_yticklabels(beh_cols)
        axes[2].set_xlabel("Time (s)")
        axes[2].set_title("Time-in-zone ethogram (fixed colors)")
        axes[2].invert_yaxis()
    axes[-1].set_xlim(fp_trim["Time_video"].min(), fp_trim["Time_video"].max())
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close(fig)

def _extract_led_traces(fp_trim: pd.DataFrame, chan: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    iso_df = fp_trim.loc[fp_trim["LedState"]==1, ["Time_video", chan]].copy()
    iso_df.rename(columns={chan: f"{chan}_iso415"}, inplace=True)
    gcamp_df = fp_trim.loc[fp_trim["LedState"]==2, ["Time_video", chan]].copy()
    gcamp_df.rename(columns={chan: f"{chan}_gcamp470"}, inplace=True)
    return iso_df.reset_index(drop=True), gcamp_df.reset_index(drop=True)

# ==========================================================
# MAIN PROCESSING LOOP
# ==========================================================

fp_traces = {}  # collect all traces

for tp in timepoints:
    tp_dir = main_dir / tp
    export_dir = tp_dir / "Export files"
    fig_dir_tp = figures_main_dir / tp
    fig_dir_raw = fig_dir_tp / "Raw_traces_ethograms"
    fig_dir_raw.mkdir(parents=True, exist_ok=True)
    
    if not tp_dir.is_dir():
        warnings.warn(f"Timepoint directory missing: {tp_dir}. Skipping.")
        continue
    
    trial_files = sorted(export_dir.glob("Raw data-*.xlsx"))
    if not trial_files:
        warnings.warn(f"No 'Raw data-*.xlsx' files found in {export_dir}")
        continue
    
    print(f"Processing timepoint: {tp}")
    
    for eth_path in trial_files:
        try:
            fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)
            geno_tag = _mouse_genotype_tag(mouse_id)
            fp_folder = tp_dir / fp_id
            if not fp_folder.is_dir():
                warnings.warn(f"FP folder not found: {fp_folder}")
                continue
            fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)
            fp = pd.read_csv(fp_csv)
            fp = fp[fp["LedState"].isin([1,2])].copy()
            ts = pd.read_csv(ts_csv)
            video_start, video_stop = _get_video_window(ts)
            fp_trim = _trim_fp_to_window(fp, video_start, video_stop)
            chan = "G0" if "G0" in fp_trim.columns else next(
                c for c in fp_trim.columns if pd.api.types.is_numeric_dtype(fp_trim[c]) and c not in {"SystemTimestamp","LedState","Time_video","Time_s","Timestamp"}
            )
            iso_df, gcamp_df = _extract_led_traces(fp_trim, chan)
            trial_stub = eth_path.stem.replace("Raw data-", "").replace(" ", "_")
            mouse_part = f"ID{mouse_id}_{geno_tag}" if geno_tag else f"ID{mouse_id}"
            file_stub = f"{mouse_part}__{fp_id}__{trial_stub}"
            key = f"{tp}__{file_stub}"
            fp_traces[key] = {"iso": iso_df, "gcamp": gcamp_df, "mouse_id": mouse_id, "genotype": geno_tag}
            eth, matched_cols = _load_ethovision_sheet_with_targets(eth_path, target_zone_labels, video_fps)
            timeline = _ethogram_segments(eth, matched_cols)
            out_png = fig_dir_raw / f"{file_stub}_zones.png"
            _plot_trial(fp_trim, timeline, matched_cols, out_png, title_prefix=f"{mouse_part} | {fp_id} | {trial_stub}")
            print(f"✅ [{tp}] {eth_path.name} → {fp_id} | Figure: {out_png.name}")
        except Exception as e:
            warnings.warn(f"[WARN] [{tp}] {eth_path.name}: {e}")

# ==========================================================
# ALL MICE GRID PLOT
# ==========================================================

def plot_all_mice_grid(fp_traces, output_png, figsize=(40,30)):
    if not fp_traces:
        warnings.warn("fp_traces is empty — no data collected. Check data folders and exported Excel files.")
        return
    parsed = []
    for key, d in fp_traces.items():
        tp, rest = key.split("__", 1)
        parsed.append({"key": key, "tp": tp, "mouse": d["mouse_id"], "geno": d["genotype"], "iso": d["iso"], "gcamp": d["gcamp"]})
    mice = sorted({p["mouse"] for p in parsed})
    timepoints_sorted = sorted({p["tp"] for p in parsed})
    lookup = {(p["mouse"], p["tp"]): p for p in parsed}
    all_vals = []
    for p in parsed:
        all_vals.extend(p["iso"].iloc[:,1].values.flatten())
        all_vals.extend(p["gcamp"].iloc[:,1].values.flatten())
    global_min, global_max = np.nanmin(all_vals), np.nanmax(all_vals)
    nrows, ncols = len(mice), len(timepoints_sorted)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, sharey=True, squeeze=False)
    for r, mouse in enumerate(mice):
        for c, tp in enumerate(timepoints_sorted):
            ax = axes[r][c]
            entry = lookup.get((mouse,tp), None)
            if entry is None:
                ax.set_axis_off()
                continue
            iso, gcamp = entry["iso"], entry["gcamp"]
            ax.plot(iso["Time_video"], iso.iloc[:,1], alpha=0.6, label="Iso", linewidth=1)
            ax.plot(gcamp["Time_video"], gcamp.iloc[:,1], alpha=0.6, label="GCaMP", linewidth=1)
            ax.set_ylim(global_min, global_max)
            if r==0:
                ax.set_title(tp, fontsize=18, pad=10)
            if c==0:
                ax.set_ylabel(f"Mouse {mouse}", fontsize=14)
            if r==nrows-1:
                ax.set_xlabel("Time (s)")
    axes[0][0].legend(loc="upper right", fontsize=12)
    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)
    print(f"Saved large grid plot → {output_png}")

big_png = figures_main_dir / "ALL_MICE__ALL_TIMEPOINTS.png"
plot_all_mice_grid(fp_traces, big_png)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings

# ==========================================================
# SETTINGS
# ==========================================================
main_dir = Path("/Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/EPM").resolve()
timepoints = ["Preinduction", "W1", "W2", "W3"]

video_fps = 50.0
fp_fps = 50.0

# Figures directory
figures_main_dir = main_dir / "Figures"
figures_main_dir.mkdir(exist_ok=True, parents=True)

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================
def _mouse_genotype_tag(mouse_id: str) -> str | None:
    id_clean = mouse_id.strip()
    if id_clean in {"372", "376", "423"}:
        return "NE"
    if id_clean in {"374", "429"}:
        return "WT"
    return None

def _read_export_and_get_fp_and_mouse_id(xl_path: Path) -> tuple[str, str]:
    df = pd.read_excel(xl_path, header=None)
    key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()
    match_fp = df[key_col == "fp file"]
    if match_fp.empty:
        raise ValueError(f"'FP file' row not found in {xl_path.name}")
    fp_id = str(df.iloc[match_fp.index[0], 1]).strip()
    match_id = df[key_col == "id"]
    if match_id.empty:
        raise ValueError(f"'ID' row not found in {xl_path.name}")
    mouse_id = str(df.iloc[match_id.index[0], 1]).strip()
    return fp_id, mouse_id

def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str) -> tuple[Path, Path]:
    fp_candidates = list(fp_folder.glob(f"{fp_id}.csv")) or [p for p in fp_folder.glob("*.csv") if fp_id.lower() in p.name.lower()]
    if not fp_candidates:
        raise FileNotFoundError(f"No FP csv for id {fp_id} in {fp_folder}")
    ts_candidates = list(fp_folder.glob("timestamp*.csv")) or [p for p in fp_folder.glob("*.csv") if "time" in p.name.lower()]
    if not ts_candidates:
        raise FileNotFoundError(f"No timestamp csv in {fp_folder}")
    return sorted(fp_candidates)[0], sorted(ts_candidates)[0]

def _coerce_bool_col(series: pd.Series) -> pd.Series:
    if series.dtype == bool: return series
    if pd.api.types.is_numeric_dtype(series): return series.astype(int).astype(bool)
    return series.astype(str).str.strip().str.lower().isin(["true","1","t","yes","y"])

def _get_video_window(ts: pd.DataFrame) -> tuple[float,float]:
    state_col = "DigitalIOState" if "DigitalIOState" in ts.columns else next(c for c in ts.columns if "digital" in c.lower() or "state" in c.lower())
    time_col = "SystemTimestamp" if "SystemTimestamp" in ts.columns else next(c for c in ts.columns if "time" in c.lower())
    state = _coerce_bool_col(ts[state_col])
    t = pd.to_numeric(ts[time_col], errors="coerce")
    if state.sum() == 0 or (~state).sum() == 0:
        raise ValueError("Timestamp must contain at least one True and one False in DigitalIOState")
    return float(t[state].iloc[0]), float(t[~state].iloc[0])

def _build_time_vector_from_fp(fp_df: pd.DataFrame) -> np.ndarray:
    if "SystemTimestamp" in fp_df.columns:
        tt = pd.to_numeric(fp_df["SystemTimestamp"], errors="coerce").to_numpy()
        if np.isfinite(tt).sum() >= len(tt)*0.8:
            return tt
    return np.arange(len(fp_df), dtype=float)/fp_fps

def _snap_down_index(t: np.ndarray, target: float) -> int:
    return max(0, np.searchsorted(t, target, side="right")-1)

def _trim_fp_to_window(fp_df: pd.DataFrame, start: float, stop: float) -> pd.DataFrame:
    tt = _build_time_vector_from_fp(fp_df)
    if len(tt) != len(fp_df):
        n = min(len(tt), len(fp_df))
        tt, fp_df = tt[:n], fp_df.iloc[:n,:].reset_index(drop=True)
    i0, i1 = _snap_down_index(tt,start), _snap_down_index(tt,stop)
    i1 = max(i1,i0)
    out = fp_df.iloc[i0:i1+1,:].copy()
    out["Time_video"] = tt[i0:i1+1]-tt[i0]
    return out

def _extract_led_traces(fp_trim: pd.DataFrame, chan: str) -> tuple[pd.DataFrame,pd.DataFrame]:
    iso = fp_trim.loc[fp_trim["LedState"]==1, ["Time_video", chan]].copy()
    iso.rename(columns={chan:f"{chan}_iso415"}, inplace=True)
    gcamp = fp_trim.loc[fp_trim["LedState"]==2, ["Time_video", chan]].copy()
    gcamp.rename(columns={chan:f"{chan}_gcamp470"}, inplace=True)
    return iso.reset_index(drop=True), gcamp.reset_index(drop=True)

# ==========================================================
# MAIN LOOP
# ==========================================================

fp_traces = {}

for tp in timepoints:
    tp_dir = main_dir / tp
    export_dir = tp_dir / "Export files"
    if not tp_dir.is_dir():
        warnings.warn(f"Timepoint directory missing: {tp_dir}. Skipping.")
        continue
    trial_files = sorted(export_dir.glob("Raw data-*.xlsx"))
    if not trial_files:
        warnings.warn(f"No Raw data-*.xlsx files in {export_dir}")
        continue

    for eth_path in trial_files:
        try:
            fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)
            geno_tag = _mouse_genotype_tag(mouse_id)
            fp_folder = tp_dir / fp_id
            if not fp_folder.is_dir():
                warnings.warn(f"FP folder missing: {fp_folder}")
                continue
            fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)
            fp = pd.read_csv(fp_csv)
            fp = fp[fp["LedState"].isin([1,2])].copy()
            ts = pd.read_csv(ts_csv)
            start, stop = _get_video_window(ts)
            fp_trim = _trim_fp_to_window(fp, start, stop)
            chan = "G0" if "G0" in fp_trim.columns else next(c for c in fp_trim.columns if pd.api.types.is_numeric_dtype(fp_trim[c]) and c not in {"SystemTimestamp","LedState","Time_video","Time_s","Timestamp"})
            iso_df, gcamp_df = _extract_led_traces(fp_trim, chan)
            file_stub = f"{tp}__ID{mouse_id}_{geno_tag or ''}__{fp_id}"
            fp_traces[file_stub] = {"iso": iso_df, "gcamp": gcamp_df, "mouse_id": mouse_id, "genotype": geno_tag, "tp": tp}
        except Exception as e:
            warnings.warn(f"[WARN] {eth_path.name}: {e}")

# ==========================================================
# ALL MICE GRID PLOT
# ==========================================================

def plot_all_mice_grid(fp_traces, output_png, figsize=(40,30)):
    if not fp_traces:
        warnings.warn("fp_traces empty — no data collected.")
        return
    parsed = list(fp_traces.values())
    mice = sorted({p["mouse_id"] for p in parsed})
    timepoints_sorted = sorted({p["tp"] for p in parsed})
    lookup = {(p["mouse_id"], p["tp"]): p for p in parsed}
    all_vals = []
    for p in parsed:
        all_vals.extend(p["iso"].iloc[:,1].values.flatten())
        all_vals.extend(p["gcamp"].iloc[:,1].values.flatten())
    global_min, global_max = np.nanmin(all_vals), np.nanmax(all_vals)

    nrows, ncols = len(mice), len(timepoints_sorted)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharey=True, squeeze=False)

    for r, mouse in enumerate(mice):
        for c, tp in enumerate(timepoints_sorted):
            ax = axes[r][c]
            entry = lookup.get((mouse,tp))
            if not entry:
                ax.set_axis_off()
                continue
            iso, gcamp = entry["iso"], entry["gcamp"]
            ax.plot(iso["Time_video"], iso.iloc[:,1], alpha=0.6, label="Iso", linewidth=1)
            ax.plot(gcamp["Time_video"], gcamp.iloc[:,1], alpha=0.6, label="GCaMP", linewidth=1)
            ax.set_ylim(global_min, global_max)
            if r==0: ax.set_title(tp, fontsize=18, pad=10)
            if c==0: ax.set_ylabel(f"Mouse {mouse}", fontsize=14)
            if r==nrows-1: ax.set_xlabel("Time (s)")
    axes[0][0].legend(loc="upper right", fontsize=12)
    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)
    print(f"Saved ALL MICE plot → {output_png}")

# Save final big plot
big_png = figures_main_dir / "ALL_MICE__ALL_TIMEPOINTS.png"
plot_all_mice_grid(fp_traces, big_png)


# Add LDCT in to plot for comparison



In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# -----------------------
# Parameters
# -----------------------
main_dir = Path("/Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/EPM")
ldct_dir = Path("/Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/LDCT/W2")
timepoints = ["Preinduction", "W1", "W2", "W3"]

video_fps = 50.0
fp_fps = 50.0

# -----------------------
# Helper functions
# -----------------------
_norm = re.compile(r"[^a-z0-9]+")

def _norm_key(s: str) -> str:
    return _norm.sub("", s.lower())

def _extract_led_traces(fp_trim: pd.DataFrame, chan: str):
    if "Time_video" not in fp_trim.columns:
        fp_trim["Time_video"] = np.arange(len(fp_trim)) / fp_fps
    iso_df = fp_trim.loc[fp_trim["LedState"] == 1, ["Time_video", chan]].copy()
    iso_df.rename(columns={chan: f"{chan}_iso415"}, inplace=True)
    gcamp_df = fp_trim.loc[fp_trim["LedState"] == 2, ["Time_video", chan]].copy()
    gcamp_df.rename(columns={chan: f"{chan}_gcamp470"}, inplace=True)
    return iso_df.reset_index(drop=True), gcamp_df.reset_index(drop=True)

def _read_export_and_get_fp_and_mouse_id(xl_path: Path):
    df = pd.read_excel(xl_path, header=None)
    key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()
    match_fp = df.loc[key_col == "fp file"]
    if match_fp.empty: raise ValueError(f"'FP file' row not found in {xl_path.name}")
    fp_id = str(match_fp.iloc[0, 1]).strip()
    match_id = df.loc[key_col == "id"]
    if match_id.empty: raise ValueError(f"'ID' row not found in {xl_path.name}")
    mouse_id = str(match_id.iloc[0, 1]).strip()
    return fp_id, mouse_id

def _mouse_genotype_tag(mouse_id: str) -> str | None:
    if mouse_id in {"372", "376", "423"}: return "NE"
    if mouse_id in {"374", "429"}: return "WT"
    return None

def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str):
    fp_csv_candidates = list(fp_folder.glob(f"{fp_id}.csv"))
    if not fp_csv_candidates:
        fp_csv_candidates = [p for p in fp_folder.glob("*.csv") if fp_id.lower() in p.name.lower()]
    if not fp_csv_candidates: raise FileNotFoundError(f"No FP csv found in {fp_folder} for id {fp_id}")
    fp_csv = sorted(fp_csv_candidates)[0]

    ts_csv_candidates = list(fp_folder.glob("timestamp*.csv"))
    if not ts_csv_candidates:
        ts_csv_candidates = [p for p in fp_folder.glob("*.csv") if "time" in p.name.lower()]
    if not ts_csv_candidates: raise FileNotFoundError(f"No timestamp csv found in {fp_folder}")
    ts_csv = sorted(ts_csv_candidates)[0]

    return fp_csv, ts_csv

# -----------------------
# Load all FP traces
# -----------------------
fp_traces = {}

# --- Normal EPM timepoints ---
for tp in timepoints:
    tp_dir = main_dir / tp
    export_dir = tp_dir / "Export files"
    if not export_dir.exists(): continue
    trial_files = sorted(export_dir.glob("Raw data-*.xlsx"))
    for eth_path in trial_files:
        fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)
        geno_tag = _mouse_genotype_tag(mouse_id)
        fp_folder = tp_dir / fp_id
        fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)
        fp = pd.read_csv(fp_csv)
        fp = fp[fp["LedState"].isin([1,2])].copy()
        chan = "G0" if "G0" in fp.columns else next(c for c in fp.columns if pd.api.types.is_numeric_dtype(fp[c]))
        iso_df, gcamp_df = _extract_led_traces(fp, chan)
        key = f"{tp}__ID{mouse_id}_{geno_tag or ''}__{fp_id}"
        fp_traces[key] = {"iso": iso_df, "gcamp": gcamp_df, "mouse_id": mouse_id, "genotype": geno_tag, "tp": tp}

# --- LDCT/W2 timepoint ---
ldct_tp = "LDCT_W2"
for fp_folder in ldct_dir.glob("EPM_FP*"):
    fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_folder.name)
    fp = pd.read_csv(fp_csv)
    fp = fp[fp["LedState"].isin([1,2])].copy()
    chan = "G0" if "G0" in fp.columns else next(c for c in fp.columns if pd.api.types.is_numeric_dtype(fp[c]))
    iso_df, gcamp_df = _extract_led_traces(fp, chan)
    mouse_id = fp_folder.name
    key = f"{ldct_tp}__{mouse_id}"
    fp_traces[key] = {"iso": iso_df, "gcamp": gcamp_df, "mouse_id": mouse_id, "genotype": None, "tp": ldct_tp}

# -----------------------
# Plotting function
# -----------------------
def plot_all_mice_grid(fp_traces, output_png, figsize=(40,30)):
    parsed = []
    for key,d in fp_traces.items():
        parsed.append({
            "key": key,
            "tp": d["tp"],
            "mouse": d["mouse_id"],
            "geno": d["genotype"],
            "iso": d["iso"],
            "gcamp": d["gcamp"],
        })

    mice_sorted = sorted({p["mouse"] for p in parsed})
    timepoints_sorted = sorted({p["tp"] for p in parsed}, key=lambda x: ["Preinduction","W1","W2","W3","LDCT_W2"].index(x))
    lookup = {(p["mouse"], p["tp"]): p for p in parsed}

    # global y-limits
    all_vals = []
    for p in parsed:
        all_vals.extend(p["iso"].iloc[:,1].values.flatten())
        all_vals.extend(p["gcamp"].iloc[:,1].values.flatten())
    global_min, global_max = np.nanmin(all_vals), np.nanmax(all_vals)

    fig, axes = plt.subplots(len(mice_sorted), len(timepoints_sorted), figsize=figsize, sharey=True, squeeze=False)
    for r, mouse in enumerate(mice_sorted):
        for c, tp in enumerate(timepoints_sorted):
            ax = axes[r][c]
            entry = lookup.get((mouse, tp), None)
            if entry is None:
                ax.set_axis_off()
                continue
            iso = entry["iso"]
            gcamp = entry["gcamp"]
            ax.plot(iso["Time_video"], iso.iloc[:,1], alpha=0.6, label="Iso", linewidth=1)
            ax.plot(gcamp["Time_video"], gcamp.iloc[:,1], alpha=0.6, label="GCaMP", linewidth=1)
            ax.set_ylim(global_min, global_max)
            if r==0: ax.set_title(tp, fontsize=18, pad=10)
            if c==0: ax.set_ylabel(f"{mouse}", fontsize=14)
            if r==len(mice_sorted)-1: ax.set_xlabel("Time (s)")
    axes[0][0].legend(loc="upper right", fontsize=12)
    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)
    print(f"Saved figure to {output_png}")

# -----------------------
# Make the big grid plot
# -----------------------
big_png = main_dir / "ALL_MICE_ALL_TIMEPOINTS_with_LDCT.png"
plot_all_mice_grid(fp_traces, big_png)


Saved figure to /Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/EPM/ALL_MICE_ALL_TIMEPOINTS_with_LDCT.png


In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings

# ==========================================================
# SETTINGS
# ==========================================================

ldct_dir = Path("/Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/LDCT/W2").resolve()
figures_ldct_dir = ldct_dir / "Figures_LDCT_W2"
figures_ldct_dir.mkdir(exist_ok=True, parents=True)

video_fps = 50.0
fp_fps = 50.0

# Fixed colors for plotting ethograms (if needed)
zone_colors = {
    "In zone 2(O1_prevC1 / Center-point)": "#E41A1C",  # red
    "In zone 2(O2_prevC2 / Center-point)": "#377EB8",  # blue
    "In zone 2(C1_prevO1 / Center-point)": "#4DAF4A",  # green
    "In zone 2(C2_prevO2 / Center-point)": "#984EA3",  # purple
    "In zone 2(NeutralZone / Center-point)": "#FF7F00",# orange
}

target_zone_labels = list(zone_colors.keys())

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

_norm = re.compile(r"[^a-z0-9]+")
def _norm_key(s: str) -> str:
    return _norm.sub("", s.lower())

def _mouse_genotype_tag(mouse_id: str) -> str | None:
    id_clean = mouse_id.strip()
    if id_clean in {"372", "376", "423"}: return "NE"
    if id_clean in {"374", "429"}: return "WT"
    return None

def _read_export_and_get_fp_and_mouse_id(xl_path: Path):
    df = pd.read_excel(xl_path, header=None)
    key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()
    match_fp = df.loc[key_col == "fp file"]
    if match_fp.empty: raise ValueError(f"'FP file' row not found in {xl_path.name}")
    fp_id = str(match_fp.iloc[0, 1]).strip()
    match_id = df.loc[key_col == "id"]
    if match_id.empty: raise ValueError(f"'ID' row not found in {xl_path.name}")
    mouse_id = str(match_id.iloc[0, 1]).strip()
    return fp_id, mouse_id

def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str):
    fp_csv = sorted(list(fp_folder.glob(f"{fp_id}.csv")))[0]
    ts_csv = sorted(list(fp_folder.glob("timestamp*.csv")))[0]
    return fp_csv, ts_csv

def _coerce_bool_col(series: pd.Series) -> pd.Series:
    if series.dtype == bool: return series
    if pd.api.types.is_numeric_dtype(series): return series.astype(int).astype(bool)
    low = series.astype(str).str.strip().str.lower()
    return low.isin(["true","1","t","yes","y"])

def _get_video_window(ts: pd.DataFrame) -> tuple[float,float]:
    state_col = "DigitalIOState" if "DigitalIOState" in ts.columns else next(
        c for c in ts.columns if "digital" in c.lower() or "state" in c.lower()
    )
    time_col = "SystemTimestamp" if "SystemTimestamp" in ts.columns else next(
        c for c in ts.columns if "time" in c.lower()
    )
    state = _coerce_bool_col(ts[state_col])
    t = pd.to_numeric(ts[time_col], errors="coerce")
    if state.sum()==0 or (~state).sum()==0:
        raise ValueError("Timestamp must contain at least one True and one False in DigitalIOState")
    video_start = float(t[state].iloc[0])
    video_stop  = float(t[~state].iloc[0])
    return video_start, video_stop

def _build_time_vector_from_fp(fp_df: pd.DataFrame) -> np.ndarray:
    if "SystemTimestamp" in fp_df.columns:
        tt = pd.to_numeric(fp_df["SystemTimestamp"], errors="coerce").to_numpy()
        if np.isfinite(tt).sum() >= len(tt)*0.8: return tt
    return np.arange(len(fp_df), dtype=float)/fp_fps

def _snap_down_index(t: np.ndarray, target: float) -> int:
    return max(0, np.searchsorted(t, target, side="right")-1)

def _trim_fp_to_window(fp_df: pd.DataFrame, start: float, stop: float) -> pd.DataFrame:
    tt = _build_time_vector_from_fp(fp_df)
    if len(tt) != len(fp_df):
        n = min(len(tt), len(fp_df))
        tt = tt[:n]
        fp_df = fp_df.iloc[:n,:].reset_index(drop=True)
    i0 = _snap_down_index(tt, start)
    i1 = _snap_down_index(tt, stop)
    i1 = max(i1, i0)
    out = fp_df.iloc[i0:i1+1,:].copy()
    out["Time_video"] = tt[i0:i1+1] - tt[i0]
    return out

def _extract_led_traces(fp_trim: pd.DataFrame, chan: str):
    iso_df = fp_trim.loc[fp_trim["LedState"]==1, ["Time_video", chan]].copy()
    iso_df.rename(columns={chan: f"{chan}_iso415"}, inplace=True)
    gcamp_df = fp_trim.loc[fp_trim["LedState"]==2, ["Time_video", chan]].copy()
    gcamp_df.rename(columns={chan: f"{chan}_gcamp470"}, inplace=True)
    return iso_df.reset_index(drop=True), gcamp_df.reset_index(drop=True)

# ==========================================================
# LOAD AND PROCESS LDCT TRACES
# ==========================================================

ldct_export_dir = ldct_dir / "Export Files"
if not ldct_export_dir.is_dir():
    raise FileNotFoundError(f"LDCT Export files not found: {ldct_export_dir}")

ldct_fp_traces = {}

trial_files = sorted(ldct_export_dir.glob("Raw data-*.xlsx"))
if not trial_files:
    warnings.warn("No LDCT Excel files found.")
else:
    for eth_path in trial_files:
        try:
            fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)
            geno_tag = _mouse_genotype_tag(mouse_id)
            fp_folder = ldct_dir / fp_id
            if not fp_folder.is_dir():
                warnings.warn(f"FP folder missing: {fp_folder}")
                continue
            fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)
            fp = pd.read_csv(fp_csv)
            fp = fp[fp["LedState"].isin([1,2])].copy()
            ts = pd.read_csv(ts_csv)
            video_start, video_stop = _get_video_window(ts)
            fp_trim = _trim_fp_to_window(fp, video_start, video_stop)
            chan = "G0" if "G0" in fp_trim.columns else next(
                c for c in fp_trim.columns if pd.api.types.is_numeric_dtype(fp_trim[c]) and c not in {"SystemTimestamp","LedState","Time_video"}
            )
            iso_df, gcamp_df = _extract_led_traces(fp_trim, chan)
            key = f"LDCT_W2__ID{mouse_id}_{geno_tag or ''}__{fp_id}"
            ldct_fp_traces[key] = {"iso": iso_df, "gcamp": gcamp_df, "mouse_id": mouse_id, "genotype": geno_tag}
        except Exception as e:
            warnings.warn(f"[LDCT] {eth_path.name}: {e}")

if not ldct_fp_traces:
    warnings.warn("No LDCT traces to plot")

# ==========================================================
# PLOT LDCT GRID
# ==========================================================

def plot_ldct_grid(fp_traces, output_png, figsize=(15,10)):
    if not fp_traces:
        warnings.warn("No traces to plot")
        return
    parsed = []
    for key,d in fp_traces.items():
        parsed.append({"key": key, "mouse": d["mouse_id"], "iso": d["iso"], "gcamp": d["gcamp"]})
    mice_sorted = sorted({p["mouse"] for p in parsed})
    lookup = {p["mouse"]: p for p in parsed}
    # global y-limits
    all_vals = []
    for p in parsed:
        all_vals.extend(p["iso"].iloc[:,1].values.flatten())
        all_vals.extend(p["gcamp"].iloc[:,1].values.flatten())
    global_min, global_max = np.nanmin(all_vals), np.nanmax(all_vals)
    nrows = len(mice_sorted)
    ncols = 1
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, sharex=True, squeeze=False)
    for r, mouse in enumerate(mice_sorted):
        ax = axes[r][0]
        entry = lookup.get(mouse)
        if entry is None:
            ax.set_axis_off()
            continue
        iso, gcamp = entry["iso"], entry["gcamp"]
        ax.plot(iso["Time_video"], iso.iloc[:,1], alpha=0.6, label="Iso", linewidth=1)
        ax.plot(gcamp["Time_video"], gcamp.iloc[:,1], alpha=0.6, label="GCaMP", linewidth=1)
        ax.set_ylabel(f"Mouse {mouse}", fontsize=12)
        ax.set_ylim(global_min, global_max)
        ax.grid(True)
    axes[-1][0].set_xlabel("Time (s)")
    axes[0][0].legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(output_png, dpi=300)
    plt.close(fig)
    print(f"Saved LDCT plot → {output_png}")

ldct_png = figures_ldct_dir / "LDCT_W2_AllMice.png"
plot_ldct_grid(ldct_fp_traces, ldct_png)


Saved LDCT plot → /Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/LDCT/W2/Figures_LDCT_W2/LDCT_W2_AllMice.png
